In [1]:
from google.colab import files

uploaded = files.upload()

Saving jedzenie_clean.csv to jedzenie_clean.csv
Saving paliwa_jak_jedzenie.csv to paliwa_jak_jedzenie.csv


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

YEARS = [str(y) for y in range(2019, 2026)]

def read_prices(path: str | Path, typ: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8-sig")
    data = df.melt(id_vars="Produkt", value_vars=YEARS, var_name="rok", value_name="cena")
    data["rok"] = data["rok"].astype(int)
    data["cena"] = pd.to_numeric(data["cena"], errors="coerce")
    data["typ"] = typ
    return data

def create_features(food_path: str | Path, fuel_path: str | Path) -> pd.DataFrame:
    df = pd.concat([read_prices(food_path, "żywność"), read_prices(fuel_path, "paliwo")], ignore_index=True)
    df = df.sort_values(["typ", "Produkt", "rok"]).reset_index(drop=True)
    g = df.groupby(["typ", "Produkt"], sort=False)["cena"]
    #cena z ubiegłego roku, niezależna od ceny z roku bieżącego.
    df["cena_lag1"] = g.shift(1)
    #zmiana procentowa r/r; informacja opisowa dla t oraz predyktor ceny t+1.
    df["zmiana_roczna_proc"] = (df["cena"] / df["cena_lag1"] - 1) * 100
    #indeks ceny względem roku bazowego 2019.
    base = df[df["rok"].eq(2019)][["typ", "Produkt", "cena"]].rename(columns={"cena":"cena_2019"})
    df = df.merge(base, on=["typ", "Produkt"], how="left", validate="many_to_one")
    df["indeks_2019_100"] = df["cena"] / df["cena_2019"] * 100
    #3-letnia średnia z cen wcześniejszych lat (nie obejmuje roku t).
    df["srednia_poprzednich_3_lat"] = df.groupby(["typ", "Produkt"], sort=False)["cena"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    #różnica dynamiki produktu względem średniej dynamiki 3 paliw.
    fuel_changes = (df[df["typ"].eq("paliwo")].groupby("rok")["zmiana_roczna_proc"]
                    .mean().rename("srednia_zmiana_paliw_proc"))
    df = df.join(fuel_changes, on="rok")
    df["roznica_dynamiki_vs_paliwa_pp"] = df["zmiana_roczna_proc"] - df["srednia_zmiana_paliw_proc"]
    #skownik opisuje tylko pięć nowych cech; kolumny pomocnicze usuwamy.
    df = df.drop(columns=["cena_2019", "srednia_zmiana_paliw_proc"])
    return df

from pathlib import Path

base = Path.cwd()

result = create_features(
    base / "jedzenie_clean.csv",
    base / "paliwa_jak_jedzenie.csv"
)

result.to_csv(
    base / "cechy_kluczowe.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    index=False
)

print(result.head(10).to_string(index=False))

    Produkt  rok    cena    typ  cena_lag1  zmiana_roczna_proc  indeks_2019_100  srednia_poprzednich_3_lat  roznica_dynamiki_vs_paliwa_pp
ONEkodiesel 2019 4.02625 paliwo        NaN                 NaN       100.000000                        NaN                            NaN
ONEkodiesel 2020 3.34900 paliwo    4.02625          -16.820863        83.179137                        NaN                      -2.923774
ONEkodiesel 2021 4.22825 paliwo    3.34900           26.254106       105.017075                   3.687625                      -0.823410
ONEkodiesel 2022 6.69225 paliwo    4.22825           58.274700       166.215461                   3.867833                      13.836137
ONEkodiesel 2023 5.12700 paliwo    6.69225          -23.388995       127.339336                   4.756500                      -6.335957
ONEkodiesel 2024 4.98450 paliwo    5.12700           -2.779403       123.800062                   5.349167                       0.011045
ONEkodiesel 2025 4.66350 paliwo   



Zdecydowano na wybór train/test split:
1. Jeden podział danych
2. Zachowana kolejność czasowa
3. Niskie ryzyko wycieku danych
4. Oceniany jest jeden wynik
5. Lepsze rozwiązanie dla danych czasowych

In [8]:
base = Path.cwd()

df = pd.read_csv(base / "cechy_kluczowe.csv", sep=";",decimal=",", encoding="utf-8-sig")

train = df[df["rok"] <= 2023].copy()
test = df[df["rok"] >= 2024].copy()

train.to_csv(base / "train.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)
test.to_csv(base / "test.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)

print("Liczba obserwacji treningowych:", len(train))
print("Liczba obserwacji testowych:", len(test))

print("\nUdział procentowy TRAIN:", round(len(train) / len(df) * 100, 2), "%")
print("Udział procentowy TEST:", round(len(test) / len(df) * 100, 2), "%")

Liczba obserwacji treningowych: 80
Liczba obserwacji testowych: 32

Udział procentowy TRAIN: 71.43 %
Udział procentowy TEST: 28.57 %


Regresja liniowa

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

train = pd.read_csv(base / "train.csv", sep=";", decimal=",", encoding="utf-8-sig")
test = pd.read_csv(base / "test.csv", sep=";", decimal=",", encoding="utf-8-sig")

target = "cena"

numeric_features = ["cena_lag1", "srednia_poprzednich_3_lat"]
categorical_features = ["Produkt", "typ"]

features = numeric_features + categorical_features

train = train.dropna(subset=[target, "cena_lag1"]).copy()
test = test.dropna(subset=[target, "cena_lag1"]).copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

# BASELINE
# Prognoza ceny na podstawie ceny z poprzedniego roku
baseline_predictions = test["cena_lag1"]

# REGRESJA LINIOWA
# Przygotowanie przetwarzania zmiennych
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

# Utworzenie pipeline
model = Pipeline(steps=[("preprocessor", preprocessor), ("regression", LinearRegression())])

model.fit(X_train, y_train)
regression_predictions = model.predict(X_test)

# Ocena
def evaluate_model(y_true, predictions, model_name):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "MAPE (%)": np.mean(np.abs((y_true - predictions) / y_true)) * 100}

results = pd.DataFrame([evaluate_model(y_test, baseline_predictions, "Baseline"),
    evaluate_model(y_test, regression_predictions, "Regresja liniowa")])

print("\nPORÓWNANIE MODELI:")
print(results.to_string(index=False))

predictions = test[["rok", "Produkt", "typ", "cena"]].copy()

predictions["baseline"] = baseline_predictions
predictions["regresja_liniowa"] = regression_predictions

predictions.to_csv(base / "predykcje.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)


PORÓWNANIE MODELI:
           Model     MAE     RMSE  MAPE (%)
        Baseline 0.71225 1.126990  7.900774
Regresja liniowa 1.04603 1.381199 13.819388


Wnioski:
1. Baseline okazał się skuteczniejszy od regresji liniowej.
2. MAE - dla baseline ceny różniły się od rzeczywistych średnio o ok. 0.71 jednostki podczas gdy dla regresji ceny różniły się już o 1.05 jednostki.
3. RMSE - przy uwzględnieniu dużych odchyleń baseline również wyszedł lepiej z mniejszym błędem
4. MAPE - dla baseline przeciętny bezwzględny błąd prognozy stanowił 7.9% rzeczywistej ceny

Dlaczego wygrał baseline?
1. Ceny z poprzednich lat dla niektórych obserwacji są dobrym przybliżeniem dla przyszłych cen. Baseline wykorzystuje ten fakt i predykcje występują z zazwyczaj małym błędem
2. Zbiór treningowy jest zbyt mały żeby regresja efektywnie się nauczyła
3. Ceny żywności i paliw są ustalane przez czynniki takie jak inflacja, koszty produkcji, ceny surowców, sytuację geopolityczną; których regresja nie obejmuje